In [13]:
import torch
import numpy as np
import random
import os
import json

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    balanced_accuracy_score
)
from torchmetrics.classification import MulticlassCalibrationError

# --- Configurazione ---
CURRENT_SEED = 17 # 11, 17, 29
COMPONENTS   = [32, 16, 8, 4]
N_CLASSES    = 4
# ----------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CURRENT_SEED)
os.makedirs('../artifacts', exist_ok=True)

print(f'Seed: {CURRENT_SEED}')
print(f'Regimi latenti: {COMPONENTS}')

Seed: 17
Regimi latenti: [32, 16, 8, 4]


In [14]:
def load_pca_split(d, seed):

    path = f'../compressedFeatures/pca_d{d}_seed_{seed}.pt'
    ckpt = torch.load(path, weights_only=False)

    X_train = ckpt['train'].numpy()
    X_val   = ckpt['val'].numpy()
    X_test  = ckpt['test'].numpy()

    # squeeze per sklearn per problemi tra medmnist (N, 1) e sklearn (N, )
    y_train = ckpt['y_train'].numpy().squeeze()
    y_val   = ckpt['y_val'].numpy().squeeze()
    y_test  = ckpt['y_test'].numpy().squeeze()

    print(f'd={d:2d} | train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')
    return X_train, X_val, X_test, y_train, y_val, y_test

for d in COMPONENTS:
    load_pca_split(d, CURRENT_SEED)

d=32 | train=(97477, 32), val=(10832, 32), test=(1000, 32)
d=16 | train=(97477, 16), val=(10832, 16), test=(1000, 16)
d= 8 | train=(97477, 8), val=(10832, 8), test=(1000, 8)
d= 4 | train=(97477, 4), val=(10832, 4), test=(1000, 4)


- **Macro-AUROC**: calcolata in multiclass one-vs-rest sulle probabilità di classe
- **Macro-F1**: calcolata sulle label ottenute via argmax
- **Balanced Accuracy**: calcolate sulle label ottenute via argmax
- **ECE**:  top-label multiclass calibration error con 15 bin uniformi e norma L1

In [15]:
def compute_metrics(y_true, y_pred, probs):

    macro_auroc = roc_auc_score(
        y_true, probs,
        multi_class='ovr',
        average='macro'
    )

    macro_f1 = f1_score(
        y_true, y_pred,
        average='macro',
        zero_division=0
    )

    bal_acc = balanced_accuracy_score(y_true, y_pred)

    ece_metric = MulticlassCalibrationError(
        num_classes=N_CLASSES,
        n_bins=15,
        norm='l1'
    )
    ece = ece_metric(
        torch.tensor(probs, dtype=torch.float32),
        torch.tensor(y_true, dtype=torch.long)
    ).item()

    return {
        'macro_auroc': round(macro_auroc, 4),
        'macro_f1':    round(macro_f1,    4),
        'bal_acc':     round(bal_acc,     4),
        'ece':         round(ece,         4)
    }

In [20]:
def run_logi_regr(X_train, y_train, X_val, y_val, X_test, y_test, d):

    clf = LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        random_state=CURRENT_SEED
    )
    clf.fit(X_train, y_train)

    # Validation
    val_probs  = clf.predict_proba(X_val)
    val_pred   = val_probs.argmax(axis=1)
    val_metrics = compute_metrics(y_val, val_pred, val_probs)

    # Test
    test_probs  = clf.predict_proba(X_test)
    test_pred   = test_probs.argmax(axis=1)
    test_metrics = compute_metrics(y_test, test_pred, test_probs)

    return clf, val_metrics, test_metrics, val_pred, test_pred, val_probs, test_probs

all_results = {}

for d in COMPONENTS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_pca_split(d, CURRENT_SEED)

    clf, val_m, test_m, val_pred, test_pred, val_probs, test_probs = run_logi_regr(X_train, y_train, X_val, y_val, X_test, y_test, d)

    all_results[d] = {
        'val':  val_m,
        'test': test_m
    }

    print(f'd={d:2d}')
    print(f'  VAL  → AUROC={val_m["macro_auroc"]:.4f} | F1={val_m["macro_f1"]:.4f} | BalAcc={val_m["bal_acc"]:.4f} | ECE={val_m["ece"]:.4f}')
    print(f'  TEST → AUROC={test_m["macro_auroc"]:.4f} | F1={test_m["macro_f1"]:.4f} | BalAcc={test_m["bal_acc"]:.4f} | ECE={test_m["ece"]:.4f}\n')

d=32 | train=(97477, 32), val=(10832, 32), test=(1000, 32)
d=32
  VAL  → AUROC=0.9500 | F1=0.7472 | BalAcc=0.7249 | ECE=0.0100
  TEST → AUROC=0.9332 | F1=0.6210 | BalAcc=0.6550 | ECE=0.1070

d=16 | train=(97477, 16), val=(10832, 16), test=(1000, 16)
d=16
  VAL  → AUROC=0.9262 | F1=0.6701 | BalAcc=0.6481 | ECE=0.0086
  TEST → AUROC=0.8869 | F1=0.4945 | BalAcc=0.5670 | ECE=0.1816

d= 8 | train=(97477, 8), val=(10832, 8), test=(1000, 8)
d= 8
  VAL  → AUROC=0.8821 | F1=0.5415 | BalAcc=0.5466 | ECE=0.0113
  TEST → AUROC=0.8483 | F1=0.3602 | BalAcc=0.4920 | ECE=0.2461

d= 4 | train=(97477, 4), val=(10832, 4), test=(1000, 4)
d= 4
  VAL  → AUROC=0.8614 | F1=0.5063 | BalAcc=0.5175 | ECE=0.0076
  TEST → AUROC=0.8373 | F1=0.3496 | BalAcc=0.4820 | ECE=0.2383



In [21]:
for d in COMPONENTS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_pca_split(d, CURRENT_SEED)

    clf, val_m, test_m, val_pred, test_pred, val_probs, test_probs = \
        run_logi_regr(X_train, y_train, X_val, y_val, X_test, y_test, d)

    artifact = {
        # identificazione run
        'd':           d,
        'seed':        CURRENT_SEED,
        'model':       'logistic_regression',
        'compression': 'pca_baseline',
        'type':        'no_quantum_ablation',

        # predizioni
        'val_y_true':  y_val.tolist(),
        'val_y_pred':  val_pred.tolist(),
        'val_probs':   val_probs.tolist(),

        'test_y_true': y_test.tolist(),
        'test_y_pred': test_pred.tolist(),
        'test_probs':  test_probs.tolist(),

        # metriche
        'val_metrics':  val_m,
        'test_metrics': test_m
    }

    save_path = f'../artifacts/no_quantum_ablation_d{d}_seed_{CURRENT_SEED}.json'
    with open(save_path, 'w') as f:
        json.dump(artifact, f, indent=2)


d=32 | train=(97477, 32), val=(10832, 32), test=(1000, 32)
d=16 | train=(97477, 16), val=(10832, 16), test=(1000, 16)
d= 8 | train=(97477, 8), val=(10832, 8), test=(1000, 8)
d= 4 | train=(97477, 4), val=(10832, 4), test=(1000, 4)
